# Практикум 05 · QR-коды и готовые модели

Ниже находится полная глава курса, затем — ячейки практики. Прочитайте главу перед изменением параметров.

Стандартный **Run All** использует учебные изображения или синтетические данные. Подготовьте Python с OpenCV и NumPy по стартовой инструкции.


# 05. Объекты, обычный QR-код и нейросетевые приложения

**Работа на нашем роботе:** создайте [личную копию курса и выберите нужное окружение](../docs/SETUP_ROBOT.md). Все команды ниже выполняйте из её корня; результаты `outputs/` будут принадлежать вашей копии. Самостоятельный поиск камеры, контейнера и ROS-топиков разобран в той же инструкции.

**Тема Yahboom:** 9.5 Opencv application. **Время:** 90 минут для базовой части; нейросетевые примеры — отдельная самостоятельная работа с подготовкой моделей по шагам 5–7.

## Цель и место выполнения

Получите измеримый результат: координаты красных объектов и текст обычного QR-кода. Затем разберите два приложения раздела производителя: оценивание позы человека OpenPose и обнаружение объектов MobileNet-SSD. Для первой части достаточно ноутбука; для нейросетевой нужны совместимые файлы модели, описания сети и собственное изображение. На ROSMASTER выполняйте всё в неподвижном состоянии, из корня курса.

Цветовой детектор и Haar добавлены как простая подготовка. Они не заменяют QR, OpenPose и SSD из раздела 9.5. Команды здесь сохраняют файлы и не включают слежение, управление колёсами или манипулятором.

## Пять понятий

**Контур** задаёт границу связной области маски. `area_px2` — геометрическая площадь контура в квадратных пикселях; она может отличаться от количества белых пикселей. `center_px` вычисляется по моментам области. Эти координаты принадлежат изображению, а расстояние в метрах требует дополнительных данных.

**Порог площади** помогает исключить маленькие цветные точки. Его значение зависит от разрешения и размера объекта в кадре. Поле `id_in_frame` — номер в данном изображении; оно не обеспечивает постоянную идентичность объекта в видео.

**QR-декодирование** восстанавливает текст, закодированный в клетках. Здесь используется `cv2.QRCodeDetector`; у производителя показан вариант с `qrcode` и `pyzbar`. Текст записывается в JSON и не выполняется как команда. Обычный QR этого урока отличается от визуальных AR-маркеров последующих занятий.

**DNN** запускает уже обученную нейросеть. Веса хранят численные параметры, конфигурация описывает вычисления, а список меток сопоставляет числовые классы с названиями. Совместимость файлов и подготовка входа важны для правильного результата.

**Поза и лицо** — разные задачи. OpenPose оценивает ключевые точки тела. Haar-каскад находит области, похожие на лица. Обнаружение лица не устанавливает личность человека. Учебный разбор OpenPose ниже выбирает максимум каждой карты для одного человека; полноценное разделение нескольких людей по PAF здесь не реализовано.

## Практика: восемь шагов

1. Посмотрите доступные режимы и найдите красные объекты:

   ```bash
   python3 -m rosmaster_cv.lesson05_applications --help
   python3 -m rosmaster_cv.lesson05_applications --input data/scene.png --out outputs/05
   ```

   В стандартной сцене ожидаются две большие красные фигуры. Маленькие красные точки отсекаются порогом площади 200.

2. Откройте `objects.png`, `red_mask.png`, `objects.json`. Сопоставьте рамки, центры и площади. Измените порог:

   ```bash
   python3 -m rosmaster_cv.lesson05_applications --min-area 20000 --out outputs/05_large
   ```

   Предскажите число объектов до запуска и объясните результат по площадям стандартного отчёта.

3. Прочитайте готовый QR из учебных данных:

   ```bash
   python3 -m rosmaster_cv.lesson05_applications --input data/qr.png --qr --out outputs/05_qr
   ```

   Откройте `qr.png` и `qr.json` в папке результата. Сверьте `text` с заданным содержимым кода `KAZNU-ROSMASTER-9.5`. `decoded: false` означает отсутствие успешно прочитанного текста, даже если границы были найдены. При необходимости код заново создаёт `python3 tools/generate_qr.py`.

4. Сфотографируйте этот код под двумя углами, сохраните снимки в `data/` и повторите команду, меняя `--input` и `--out`. Запишите расстояние приблизительно, наличие размытия и факт успешного чтения. Файлы собственных снимков вы создаёте сами.

5. На образе ROSMASTER найдите исходники производителя:

   ```bash
   ls ~/yahboomcar_ws/src/yahboomcar_visual/simple_qrcode
   ls ~/yahboomcar_ws/src/yahboomcar_visual/detection
   ```

   Если указанных каталогов нет, на host выполните `find "$HOME/temp/yahboomcar_ws" "$HOME/yahboomcar_ws" -type d -name detection 2>/dev/null`; в контейнере проверьте `/root/yahboomcar_ws/src/yahboomcar_visual/detection`. Используйте найденный путь вместо примера ниже. Откройте файлы редактором, например `nano ПУТЬ_К_ФАЙЛУ`. Выпишите имена моделей и параметры `blobFromImage`. На странице Yahboom команда `python target_detection.py` повторяется в блоках позы и объектов; конкретный файл для OpenPose нужно определить по установленному исходнику. Наличие папки зависит от версии образа. Отсутствие каталога запишите в отчёте, затем используйте [официальный раздел](https://www.yahboom.net/study/ROSMASTER-X3-PLUS/8146/), где размещены ссылки на Code.

6. **SSD при наличии файлов.** Подготовьте `data/person.jpg` — собственную фотографию человека или другой категории COCO. Для подтверждённого примера Yahboom используется TensorFlow MobileNet v2:

   ```bash
   python3 -m rosmaster_cv.lesson05_applications --input data/person.jpg --ssd-model "$HOME/yahboomcar_ws/src/yahboomcar_visual/detection/frozen_inference_graph.pb" --ssd-config "$HOME/yahboomcar_ws/src/yahboomcar_visual/detection/ssd_mobilenet_v2_coco.txt" --ssd-labels "$HOME/yahboomcar_ws/src/yahboomcar_visual/detection/object_detection_coco.txt" --confidence 0.4 --out outputs/05_ssd
   ```

   Сначала проверьте наличие этих файлов; если они лежат иначе, замените пути. Программа использует подготовку входа из примера производителя: 300×300, mean 104/117/123, swapRB. Список классов должен сохранять соответствие `class_id−1`, включая пропуски COCO. Сравните пороги `--confidence 0.4` и `0.7`.

7. **Поза при наличии модели.** В отдельном авторском примере применяются Caffe COCO18 или MPI15. Совместимую пару получают из [официальных материалов OpenPose](https://github.com/CMU-Perceptual-Computing-Lab/openpose/tree/master/models), с учётом условий использования. Для выбранной COCO18-пары:

   ```bash
   python3 -m rosmaster_cv.lesson05_applications --input data/person.jpg --pose-prototxt models/pose_deploy_linevec.prototxt --pose-model models/pose_iter_440000.caffemodel --pose-format coco --out outputs/05_pose
   ```

   Это имена выбранной пары, а не обещание её наличия в комплекте. Сохраните собственные пути при необходимости. Исследуйте фото с одним человеком в полный рост. Параметр `--pose-threshold` управляет выводом точек. CPU-выполнение на Jetson может быть медленным; движение робота от результата не зависит.

8. **Дополнительный опыт с лицами:**

   ```bash
   python3 -m rosmaster_cv.lesson05_applications --input data/person.jpg --faces --out outputs/05_faces
   ```

   Оцените пропуски и ложные рамки. При отсутствии установленного каскада задайте `--cascade` с путём к `haarcascade_frontalface_default.xml`.

## Файлы, самостоятельное задание и сдача

Всегда создаются `objects.png`, `red_mask.png`, `objects.json`, `report.json`. Режимы добавляют пары `qr.png/qr.json`, `ssd.png/ssd.json`, `pose.png/pose.json`, `faces.png/faces.json`.

Сдайте таблицу площадей и центров, два опыта с QR, сравнение двух порогов и ответы ниже. Для нейросетевой части приложите имена моделей, параметры и результаты; при отсутствии весов честно укажите «не выполнено: нет модели» и приложите разбор исходника. Это не считается успешным запуском сети. Веса не включены в учебный архив, их инференс без соответствующих файлов не проверяется.

## Три вопроса и ошибки

1. Почему центр в пикселях не задаёт положение предмета относительно робота в метрах?
2. Чем текст QR отличается от числового ID AR-маркера?
3. Почему рамка лица не подтверждает личность человека?

**Модель не найдена:** проверьте пути, формат и совместимость пары; программа не загружает веса автоматически. **QR не читается:** обеспечьте резкость, видимость всего кода и светлого поля вокруг; попробуйте фронтальный снимок.

Источник структуры: [Yahboom 9.5](https://www.yahboom.net/study/ROSMASTER-X3-PLUS/8146/). Далее: [06 — AR-визуализация](../lessons/06_ar.md).


---

## Ячейки практики

Запускайте последовательно. Команды используют тот же Python, что и ядро ноутбука. Рабочая папка дочерних процессов — корень репозитория. Ошибка команды останавливает ячейку и показывает причину.


In [ ]:
from pathlib import Path
import subprocess
import sys

# Jupyter may start in the repository root or in its notebooks directory.
ROOT = None
for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    if (candidate / "rosmaster_cv").is_dir() and (candidate / "lessons").is_dir():
        ROOT = candidate
        break
if ROOT is None:
    raise RuntimeError("Откройте ноутбук из корня репозитория или его папки notebooks.")

def run_module(module, *arguments):
    command = [sys.executable, "-m", "rosmaster_cv." + module]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

def run_tool(filename, *arguments):
    command = [sys.executable, str(ROOT / "tools" / filename)]
    command.extend(str(argument) for argument in arguments)
    return subprocess.run(command, cwd=str(ROOT), check=True)

print("Корень курса:", ROOT)
print("Python ядра:", sys.executable)


In [ ]:
run_tool("generate_data.py")
run_module("lesson05_applications", "--input", "data/scene.png", "--out", "outputs/05")


In [ ]:
def show_results(relative_directory, limit=4):
    directory = ROOT / relative_directory
    paths = sorted(directory.glob("*.png"))[:limit]
    if not paths:
        print("Изображений пока нет:", directory)
        return
    try:
        from IPython.display import Image, display
    except ImportError:
        for path in paths:
            print(path)
        return
    for path in paths:
        print(path.relative_to(ROOT))
        display(Image(filename=str(path)))

show_results("outputs/05")


## Результат самостоятельной работы

Выполните задания и критерии приёмки из главы. Сохраните созданные изображения и JSON, параметры запуска и краткие ответы в своём отчёте. Повторите опыт с собственными данными после успешного учебного запуска.
